In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
grid_capacity = pd.read_parquet(
    "../data/interim/weather_processed/grid_capacity.parquet"
)

In [ ]:
grid_capacity["gvws_weight"] = (
    grid_capacity["Installed Capacity (MWelec)"]
    /
    grid_capacity["Installed Capacity (MWelec)"].sum()
)

In [ ]:
grid_capacity.to_csv(
    "../data/qgis/grid_capacity.csv",
    index=False
)

In [ ]:
capacity_gdf = gpd.GeoDataFrame(
    grid_capacity,
    geometry=gpd.points_from_xy(
        grid_capacity["era5_lon"],
        grid_capacity["era5_lat"]
    ),
    crs="EPSG:4326"
)

In [ ]:
cap_curve = grid_capacity.sort_values(
    "Installed Capacity (MWelec)",
    ascending=False
).copy()

cap_curve["cum_capacity"] = (
    cap_curve["Installed Capacity (MWelec)"]
    .cumsum()
)

cap_curve["cum_capacity_pct"] = (
    cap_curve["cum_capacity"]
    /
    cap_curve["Installed Capacity (MWelec)"].sum()
    * 100
)

cap_curve["cell_pct"] = (
    np.arange(1, len(cap_curve) + 1)
    /
    len(cap_curve)
    * 100
)

In [ ]:
plt.figure(figsize=(9,6))

plt.plot(
    cap_curve["cell_pct"],
    cap_curve["cum_capacity_pct"],
    linewidth=3
)

plt.axhline(
    y=50,
    color="grey",
    linestyle="--",
    alpha=0.5
)

plt.axhline(
    y=80,
    color="grey",
    linestyle="--",
    alpha=0.5
)

plt.xlabel("% of Active ERA5 Cells")
plt.ylabel("% of Installed Capacity")

plt.title(
    "Cumulative Distribution of Installed Wind Capacity Across Active ERA5 Cells"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()

#plt.savefig(
#    "../data/results/cumulative_capacity_curve.png",
#    dpi=300
#)

plt.show()

In [ ]:
uk = gpd.read_file(
    "../data/raw/cornie/cornie.shp"
)

uk = uk.to_crs("EPSG:4326")

In [ ]:
capacity_gdf["gvws_pct"] = (
    capacity_gdf["gvws_weight"] * 100
)

In [ ]:
major_cells = capacity_gdf[
    capacity_gdf["gvws_weight"] >=
    capacity_gdf["gvws_weight"].quantile(0.90)
]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 14))

# 2. Bold, Solid Basemap (No Borders)
uk.plot(
    ax=ax,
    color="#8fc0b3", 
    edgecolor="none" 
)

capacity_gdf_sorted = capacity_gdf.sort_values(by="gvws_pct", ascending=False)

# 3. High-Opacity Data Overlay
capacity_gdf_sorted.plot(
    ax=ax,
    column="gvws_weight",
    cmap="inferno",           
    markersize=capacity_gdf_sorted["gvws_pct"] * 250,
    alpha=0.85,            
    legend=True,
    legend_kwds={
        "label": "GVWS Influence (%)",
        "shrink": 0.7
    }
)

# 4. Typography Adapted for Dark Theme
plt.title(
    "Spatial Distribution of GVWS Weights Across UK ERA5 Cells",
    fontsize=16,
    pad=20,
    fontweight="bold",
    color="white"
)
plt.xlabel("Longitude", fontsize=12, color="white")
plt.ylabel("Latitude", fontsize=12, color="white")

# Adjust tick colors and grid lines
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('white')
ax.spines['left'].set_color('white')

plt.tight_layout()
plt.savefig("../data/results/GVWS_weight_map_bold.png", dpi=300, bbox_inches="tight", facecolor='#121212')
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(12,14)
)

uk.plot(
    ax=ax,
    color="whitesmoke",
    edgecolor="lightblue",
    linewidth=0.5
)

capacity_gdf.plot(
    ax=ax,

    column="gvws_weight",

    cmap="plasma",

    markersize=
        capacity_gdf["gvws_pct"] * 200,

    alpha=0.8,

    legend=True,

    legend_kwds={
        "label":
        "GVWS Infleunce (%)"
    }
)

plt.title(
    "Spatial Distribution of GVWS Weights Across UK ERA5 Cells",
    fontsize=14
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    "../data/results/GVWS_weight_map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()